# Chapter 4: Implementing a GPT Model from Scratch

This notebook combines everything to build a complete GPT model:
- Transformer blocks (multi-head attention + feed-forward)
- Complete GPT architecture
- Model initialization and configuration
- Forward and backward pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 4.1: Layer Normalization

In [ ]:
# PyTorch's LayerNorm is already optimized, but let's understand it
class LayerNormalization(nn.Module):
    """Layer normalization: normalize over feature dimension"""
    
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta = nn.Parameter(torch.zeros(emb_dim))
    
    def forward(self, x):
        # x shape: (batch, seq_len, emb_dim)
        
        # Compute mean and variance over the embedding dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # Scale and shift
        return self.gamma * x_norm + self.beta

print("LayerNormalization implemented (comparing with PyTorch's nn.LayerNorm)")

## 4.2: Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network in transformer"""
    
    def __init__(self, emb_dim, hidden_dim, dropout=0.1):
        super().__init__()
        # Two linear layers with activation in between
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()
    
    def forward(self, x):
        """
        x: (batch, seq_len, emb_dim)
        """
        # Expand dimension
        x = self.fc1(x)  # (batch, seq_len, hidden_dim)
        
        # Apply activation
        x = self.activation(x)
        
        # Dropout
        x = self.dropout(x)
        
        # Project back
        x = self.fc2(x)  # (batch, seq_len, emb_dim)
        
        # Dropout
        x = self.dropout(x)
        
        return x

print("FeedForward network implemented")

## 4.3: Transformer Block

class TransformerBlock(nn.Module):
    """Single transformer block with multi-head attention and feed-forward"""
    
    def __init__(self, emb_dim, num_heads, hidden_dim, dropout=0.1, device=None):
        super().__init__()
        
        # Multi-head attention
        self.attention = MultiHeadAttention(emb_dim, num_heads, dropout=dropout, device=device)
        
        # Feed-forward network
        self.ffn = FeedForward(emb_dim, hidden_dim, dropout=dropout)
        
        # Layer normalization (applied before each sub-layer in Pre-LN architecture)
        self.norm1 = nn.LayerNorm(emb_dim, device=device)
        self.norm2 = nn.LayerNorm(emb_dim, device=device)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, emb_dim)
        mask: Optional causal mask
        """
        # Sub-layer 1: Multi-head attention with residual connection
        x_norm = self.norm1(x)
        attn_out = self.attention(x_norm, mask=mask)
        attn_out = self.dropout(attn_out)
        x = x + attn_out  # Residual connection
        
        # Sub-layer 2: Feed-forward with residual connection
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        ffn_out = self.dropout(ffn_out)
        x = x + ffn_out  # Residual connection
        
        return x

print("TransformerBlock implemented")

## 4.4: GPT Model

In [ ]:
class GPTModel(nn.Module):
    """GPT-like language model"""
    
    def __init__(self, cfg, device=None):
        super().__init__()
        self.cfg = cfg
        self.device = device or torch.device('cpu')
        
        # Token and position embeddings
        self.token_embedding = nn.Embedding(
            cfg['vocab_size'], 
            cfg['emb_dim'],
            device=device
        )
        self.pos_embedding = nn.Embedding(
            cfg['context_length'],
            cfg['emb_dim'],
            device=device
        )
        
        self.dropout = nn.Dropout(cfg['dropout'])
        
        # Stack of transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    emb_dim=cfg['emb_dim'],
                    num_heads=cfg['n_heads'],
                    hidden_dim=cfg['emb_dim'] * 4,  # Common: 4x expansion
                    dropout=cfg['dropout'],
                    device=device
                )
                for _ in range(cfg['n_layers'])
            ]
        )
        
        # Final layer norm
        self.norm = nn.LayerNorm(cfg['emb_dim'], device=device)
        
        # Output projection to vocabulary
        self.output_proj = nn.Linear(
            cfg['emb_dim'],
            cfg['vocab_size'],
            bias=False,
            device=device
        )
        
        # Create causal mask
        self.register_buffer(
            'causal_mask',
            self._create_causal_mask(cfg['context_length'])
        )
    
    def _create_causal_mask(self, seq_len):
        """Create lower triangular mask for causal attention"""
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        mask = (1 - mask).bool()
        # Expand for batch and head dimensions: (1, 1, seq_len, seq_len)
        return mask.unsqueeze(0).unsqueeze(0)
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch, seq_len) tensor of token IDs
        
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        
        # Token embeddings
        token_embs = self.token_embedding(input_ids)  # (batch, seq_len, emb_dim)
        
        # Position embeddings
        pos_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
        pos_embs = self.pos_embedding(pos_ids)  # (1, seq_len, emb_dim)
        
        # Combine embeddings
        x = token_embs + pos_embs
        x = self.dropout(x)
        
        # Apply causal mask
        mask = self.causal_mask[:, :, :seq_len, :seq_len]
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x, mask=mask)
        
        # Final layer norm
        x = self.norm(x)
        
        # Project to vocabulary
        logits = self.output_proj(x)  # (batch, seq_len, vocab_size)
        
        return logits

print("GPTModel implemented")

## 4.5: Model Configuration and Initialization

In [ ]:
# Small model for testing (can be increased for better performance)
GPT_SMALL = {
    'vocab_size': 50257,      # GPT-2 vocab size
    'context_length': 256,    # Maximum sequence length
    'emb_dim': 256,           # Embedding dimension
    'n_heads': 8,             # Number of attention heads
    'n_layers': 6,            # Number of transformer blocks
    'dropout': 0.1
}

# Medium model for better results
GPT_MEDIUM = {
    'vocab_size': 50257,
    'context_length': 512,
    'emb_dim': 768,
    'n_heads': 12,
    'n_layers': 12,
    'dropout': 0.1
}

# Large model (similar to GPT-2 base)
GPT_LARGE = {
    'vocab_size': 50257,
    'context_length': 1024,
    'emb_dim': 1024,
    'n_heads': 16,
    'n_layers': 24,
    'dropout': 0.1
}

print("Model configurations defined")

In [ ]:
# Create a small model
model = GPTModel(GPT_SMALL, device=device).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Configuration: GPT_SMALL")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel device: {next(model.parameters()).device}")

# Estimate memory usage
memory_mb = (total_params * 4) / (1024 * 1024)  # 4 bytes per float32
print(f"Estimated model size: {memory_mb:.2f} MB")

## 4.6: Model Forward Pass

In [ ]:
# Test forward pass
batch_size = 4
seq_len = 64

# Create dummy input
input_ids = torch.randint(0, GPT_SMALL['vocab_size'], (batch_size, seq_len), device=device)

print(f"Input shape: {input_ids.shape}")
print(f"Input sample (first sequence, first 20 tokens): {input_ids[0, :20].tolist()}")

# Forward pass
with torch.no_grad():
    logits = model(input_ids)

print(f"\nOutput logits shape: {logits.shape}")
print(f"  (batch_size={batch_size}, seq_len={seq_len}, vocab_size={GPT_SMALL['vocab_size']})")

# Analyze logits
print(f"\nLogits statistics:")
print(f"  Min: {logits.min().item():.4f}")
print(f"  Max: {logits.max().item():.4f}")
print(f"  Mean: {logits.mean().item():.4f}")
print(f"  Std: {logits.std().item():.4f}")

## 4.7: Loss Computation

In [ ]:
# Prepare inputs and targets for loss computation
input_ids = torch.randint(0, GPT_SMALL['vocab_size'], (batch_size, seq_len), device=device)
target_ids = torch.randint(0, GPT_SMALL['vocab_size'], (batch_size, seq_len), device=device)

# Forward pass
logits = model(input_ids)  # (batch, seq_len, vocab_size)

# Compute loss
# Reshape for cross-entropy loss
logits_flat = logits.reshape(-1, GPT_SMALL['vocab_size'])  # (batch*seq_len, vocab_size)
targets_flat = target_ids.reshape(-1)  # (batch*seq_len,)

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits_flat, targets_flat)

print(f"Loss: {loss.item():.4f}")
print(f"\nInterpretation:")
print(f"  Random baseline loss: {math.log(GPT_SMALL['vocab_size']):.4f}")
print(f"  (log of vocab size for uniform distribution)")

## 4.8: Model Architecture Summary

In [ ]:
print("=" * 60)
print("GPT Model Architecture Summary")
print("=" * 60)
print()
print(model)
print()
print("=" * 60)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print("=" * 60)

## 4.9: Parameter Initialization

In [ ]:
def init_weights(model):
    """Initialize model weights using standard approaches"""
    for name, param in model.named_parameters():
        if 'weight' in name:
            if param.dim() >= 2:
                # Xavier initialization for weights
                nn.init.xavier_uniform_(param)
            else:
                nn.init.normal_(param, std=0.02)
        elif 'bias' in name:
            nn.init.constant_(param, 0)
        elif 'embedding' in name:
            nn.init.normal_(param, std=0.02)

# Create and initialize a new model
model = GPTModel(GPT_SMALL, device=device).to(device)
init_weights(model)

print("Model weights initialized")
print(f"\nSample weight statistics:")
for name, param in list(model.named_parameters())[:3]:
    print(f"  {name}: mean={param.mean().item():.6f}, std={param.std().item():.6f}")

## Summary

You've now implemented a complete GPT model! Key components:

1. **LayerNorm**: Normalizes activations for stable training
2. **Feed-Forward**: Position-wise fully connected network
3. **TransformerBlock**: Combines attention and feed-forward with residuals
4. **GPTModel**: Full model with embeddings, stacked blocks, and output projection

The model can now be trained using standard PyTorch training loops!